In [3]:
# for colab setup
!nvidia-smi

Mon Nov 16 10:57:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
%%bash
# mini-infer Colab source sync and CUDA Toolkit check
set -e
if command -v nvcc >/dev/null 2>&1; then
  nvcc --version | tail -n 4
else
  echo "nvcc not found; nvidia-smi reports driver support, not the installed Toolkit"
fi
repo_dir="${MINI_INFER_DIR:-$PWD/mini-infer}"
if [ ! -d "$repo_dir/.git" ]; then
  git clone --depth 1 https://github.com/szfmsmdx/mini-infer.git "$repo_dir"
else
  if [ -n "$(git -C "$repo_dir" status --porcelain)" ]; then
    echo "Repository has local changes; resolve them before updating."
    exit 1
  fi
  git -C "$repo_dir" fetch origin main
  git -C "$repo_dir" switch --detach origin/main
fi
git -C "$repo_dir" rev-parse --short HEAD


Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
2b67833


From https://github.com/szfmsmdx/mini-infer
 * branch            main       -> FETCH_HEAD
HEAD is now at 2b67833 [docs] 调整任务完成后的提交和推送规则


In [5]:
%%bash
# mini-infer Colab CUDA build (NVIDIA L4, SM89)
set -e
repo_dir="${MINI_INFER_DIR:-$PWD/mini-infer}"
build_dir="$repo_dir/build/cuda-colab"
cuda_arch="${CUDA_ARCH:-80}"
cmake -S "$repo_dir" -B "$build_dir" -DBUILD_TESTING=ON "-DCMAKE_CUDA_ARCHITECTURES=$cuda_arch"
cmake --build "$build_dir" -j 2


-- CUDA enabled: 13.0.88
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /content/mini-infer/build/cuda-colab
[  2%] Building CXX object CMakeFiles/mini_infer.dir/autograd/autograd.cpp.o
[  4%] Building CXX object CMakeFiles/mini_infer.dir/autograd/backward_nodes/elementwise.cpp.o
[  6%] Building CXX object CMakeFiles/mini_infer.dir/autograd/backward_nodes/reduction.cpp.o
[  9%] Building CXX object CMakeFiles/mini_infer.dir/autograd/backward_nodes/tensor_ops.cpp.o
[ 11%] Building CXX object CMakeFiles/mini_infer.dir/ops/tensor_ops.cpp.o
[ 13%] Building CXX object CMakeFiles/mini_infer.dir/ops/elementwise.cpp.o
[ 16%] Building CXX object CMakeFiles/mini_infer.dir/ops/reduction.cpp.o
[ 18%] Building CXX object CMakeFiles/mini_infer.dir/ops/cpu/elementwise.cpp.o
[ 20%] Building CXX object CMakeFiles/mini_infer.dir/ops/cpu/reduction.cpp.o
[ 23%] Building CXX object CMakeFiles/mini_infer.dir/tensor/tensor.cpp.o
[ 25%] Building CXX object CMakeFiles/

In [6]:
%%bash
# mini-infer Colab tests (including GPU Tensor)
set -e
repo_dir="${MINI_INFER_DIR:-$PWD/mini-infer}"
ctest --test-dir "$repo_dir/build/cuda-colab" --output-on-failure


Internal ctest changing into directory: /content/mini-infer/build/cuda-colab
Test project /content/mini-infer/build/cuda-colab
      Start  1: reduction_cpu_test
 1/13 Test  #1: reduction_cpu_test ...............   Passed    0.00 sec
      Start  2: layout_cpu_test
 2/13 Test  #2: layout_cpu_test ..................   Passed    0.00 sec
      Start  3: scalar_test
 3/13 Test  #3: scalar_test ......................   Passed    0.00 sec
      Start  4: autograd_cpu_test
 4/13 Test  #4: autograd_cpu_test ................   Passed    0.00 sec
      Start  5: logger_test
 5/13 Test  #5: logger_test ......................   Passed    0.00 sec
      Start  6: dtype_encode_test
 6/13 Test  #6: dtype_encode_test ................   Passed    0.00 sec
      Start  7: tensor_cpu_test
 7/13 Test  #7: tensor_cpu_test ..................   Passed    0.00 sec
      Start  8: elementwise_cpu_test
 8/13 Test  #8: elementwise_cpu_test .............   Passed    0.00 sec
      Start  9: reduction_cuda_test
 